# 04 · Validate — calibration study (ROC/PR, composite, agreement)

**Standard slot:** *validate (in silico).* **For Project 01 this is the core science:** which metric
actually predicts the *known* experimental outcome, at what cutoff, and how do the three predictors
agree? (D3.)

Needs `results/predictions.csv` with a real `outcome` column (success/fail).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · ROC / PR per metric

Sweep each metric's threshold and measure separation against the true outcome. Report AUC. PR is
more informative than ROC when successes are rare. **Note metric direction:** higher pLDDT = better,
but lower PAE and lower scRMSD = better, so flip their sign before scoring.

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

pred = pd.read_csv("results/predictions.csv")
# Collapse to one row per design using a chosen tool (prefer AF2).
order = {"af2":0,"esmfold":1,"boltz":2,"mock":3}
pred["pri"] = pred["tool"].map(order).fillna(9)
one = pred.sort_values("pri").groupby("id", as_index=False).first()
y = (one["outcome"].astype(str).str.lower() == "success").astype(int).values

# Higher-is-better score per metric (flip PAE).
metrics = {}
if one["plddt"].notna().any(): metrics["pLDDT"] = one["plddt"].values
if one["pae"].notna().any():   metrics["-PAE"]  = -one["pae"].values
if one["ptm"].notna().any():   metrics["pTM"]   = one["ptm"].values

def safe_auc(yv, s):
    m = ~pd.isna(s)
    return roc_auc_score(yv[m], np.asarray(s)[m]) if len(set(yv[m])) > 1 else float("nan")

print("AUC by single metric:")
for name, s in metrics.items():
    print(f"  {name:6s} ROC-AUC={safe_auc(y, s):.3f}  PR-AUC={average_precision_score(y, np.nan_to_num(s)):.3f}")

In [ ]:
# ROC curves (only meaningful with a real labeled dataset; trivial on the 2-row seed).
plt.figure(figsize=(5,5))
for name, s in metrics.items():
    m = ~pd.isna(s)
    if len(set(y[m])) < 2:
        continue
    fpr, tpr, _ = roc_curve(y[m], np.asarray(s)[m])
    plt.plot(fpr, tpr, label=f"{name}")
plt.plot([0,1],[0,1],"k--",lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("Metric ROC vs experimental outcome"); plt.legend()
plt.tight_layout(); plt.savefig("results/roc_metrics.png", dpi=150); plt.show()

## 2 · Composite predictor `[extension]`

Fit a simple logistic regression on the standardized metrics and compare its cross-validated AUC to
the best single metric. Keep it to a few features and **report N** — with a small dataset this
overfits easily; use cross-validation and report a confidence interval (bootstrap).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

feat_cols = [c for c in ["plddt","pae","ptm"] if one[c].notna().all()]
if len(feat_cols) >= 2 and len(set(y)) > 1 and len(y) >= 10:
    X = one[feat_cols].values
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    auc = cross_val_score(clf, X, y, cv=min(5, sum(y), sum(1-y)), scoring="roc_auc")
    print(f"composite CV ROC-AUC = {auc.mean():.3f} ± {auc.std():.3f}  (features={feat_cols}, N={len(y)})")
else:
    print("Need a real labeled dataset (N>=10, both classes, >=2 complete features) for the composite model.")

## 3 · AF2 / ESMFold / Boltz agreement

Correlate the predictors' pLDDT (and scRMSD once computed). Disagreement maps the harness's blind
spots — often novel folds or MSA-poor sequences. Flag and discuss them.

In [ ]:
wide = pred.pivot_table(index="id", columns="tool", values="plddt")
if wide.shape[1] >= 2:
    print("pLDDT correlation across tools:")
    print(wide.corr().round(2))
    import matplotlib.pyplot as plt
    pd.plotting.scatter_matrix(wide, figsize=(6,6)); plt.suptitle("Cross-tool pLDDT"); plt.tight_layout()
    plt.savefig("results/agreement.png", dpi=150); plt.show()
else:
    print("Run >=2 tools to assess agreement.")

## D3 (part 2) checklist
- [ ] ROC/PR + AUC for every metric, with N and (ideally) bootstrap CIs.
- [ ] Best single metric identified; composite model compared honestly.
- [ ] Cross-tool agreement reported; disagreement cases flagged.
- [ ] Failure-mode table: where does the best predictor get it wrong, and on what kind of design?

**Next:** `05_validation_plan.ipynb` — turn this into the cohort SOP.